In [1]:
!pip install -q openai

In [2]:
import json
import os
import queue
import subprocess
import threading
import time
import uuid
from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum, auto
from pathlib import Path
from typing import Callable, Optional
from openai import OpenAI

os.environ['OPENAI_API_KEY'] = ''
client = OpenAI()
MODEL = 'gpt-5-mini'

In [14]:
#  === Minimal agent loop (self-contained copy from agent_core.ipynb) ===
def run_agent_loop(messages, tools, tool_handlers,
                   system='', max_iterations=20, on_tool_call=None):
    # messages: list[dict] (N,) -- mutated in-place
    full = ([{'role':'system','content':system}] + messages) if system else list(messages)
    for _ in range(max_iterations):
        resp   = client.chat.completions.create(
            model=MODEL, messages=full,
            tools=tools if tools else None,
            tool_choice='auto' if tools else None,
        )
        msg    = resp.choices[0].message
        reason = resp.choices[0].finish_reason
        ser    = {'role':'assistant','content':msg.content}
        if msg.tool_calls:
            ser['tool_calls'] = [
                {'id':tc.id,'type':'function',
                 'function':{'name':tc.function.name,'arguments':tc.function.arguments}}
                for tc in msg.tool_calls]
        full.append(ser); messages.append(ser)
        if reason != 'tool_calls': return messages
        for tc in msg.tool_calls:
            name = tc.function.name
            inp  = json.loads(tc.function.arguments)
            if on_tool_call: on_tool_call(name, inp)
            h = tool_handlers.get(name)
            try:    result = h(**inp) if h else f'[ToolError] No handler: {name}'
            except Exception as exc: result = f'[ToolError] {type(exc).__name__}: {exc}'
            tm = {'role':'tool','tool_call_id':tc.id,'content':str(result)}
            full.append(tm); messages.append(tm)
    return messages

print(f'Client ready. Model: {MODEL}')

Client ready. Model: gpt-5-mini


# 1) Intelligence Layer

Think of it like a person introduced to a new task:
1. What kind of entity am I? (BASE)
2. Who am I specifically? (IDENTITY)
3. What are my values and personality? (SOUL)
4. What do I remember about this user? (MEMORY)
5. What special skills have I loaded? (SKILLS)
6. What tools do I have? (TOOLS)
7. What is happening right now? (CONTEXT)
8. Am I in a proactive heartbeat cycle? (HEARTBEAT)

Each layer is a separate file on disk. Swapping files changes the agent personality without touching code -- exactly how OpenClaw configs work.

| Layer | File | Cached? | Changes when |
|-------|------|---------|--------------|
| BASE | hardcoded | yes | never |
| IDENTITY | IDENTITY.md | yes | agent role changes |
| SOUL | SOUL.md | yes | personality update |
| MEMORY | MEMORY.md | no | every turn |
| SKILLS | skills/*.md | yes | skill loaded |
| TOOLS | TOOLS.md | yes | tool set changes |
| CONTEXT | dynamic | no | every turn |
| HEARTBEAT | HEARTBEAT.md | yes | heartbeat mode |

```python
assembler = PromptAssembler(workspace='workspace/')
assembler.load_soul('soul_friendly.md')
assembler.set_memory('User prefers concise answers. Dislikes jargon.')
prompt = assembler.build(mode='heartbeat')
# -> 8-section prompt string, ~1200 tokens
```

In [15]:
# === Workspace seed ===
# Stores all personality/memory files under workspace/.

WORKSPACE = Path('workspace')
WORKSPACE.mkdir(exist_ok=True)

_SEEDS = {
    'IDENTITY.md': (
        '# Identity\n'
        'You are Claw, a proactive AI assistant embedded in a messaging gateway.\n'
        'You have access to tools and can take actions on behalf of users.\n'
        'You communicate across Telegram, Feishu, and WebSocket channels.\n'
    ),
    'SOUL.md': (
        '# Soul\n'
        '## Values\n'
        '- Be concise. Say exactly what is needed, no more.\n'
        '- Be proactive. Notice things the user has not asked about yet.\n'
        '- Be honest. If you do not know, say so.\n'
        '## Tone\n'
        'Direct but warm. Avoid corporate language and filler phrases.\n'
        '## Boundaries\n'
        'Do not pretend to have feelings. Do not roleplay as a human.\n'
    ),
    'TOOLS.md': (
        '# Tools\n'
        '## Available\n'
        'bash: run shell commands\n'
        'file_read: read file contents\n'
        'file_write: write files\n'
        '## Policy\n'
        'Always prefer read-only operations. Confirm before destructive writes.\n'
    ),
    'HEARTBEAT.md': (
        '# Heartbeat Mode\n'
        'You are running in a scheduled heartbeat cycle, not responding to a user message.\n'
        '## Goal\n'
        'Review the current context and decide if there is anything worth doing proactively.\n'
        '## Instructions\n'
        '- Check for pending tasks or reminders.\n'
        '- If nothing needs attention, respond with exactly: IDLE\n'
        '- Otherwise, take the action and summarise what you did.\n'
    ),
    'MEMORY.md': (
        '# Memory\n'
        '- User prefers short answers\n'
        '- Timezone: Asia/Singapore (UTC+8)\n'
    ),
}

for fname, content in _SEEDS.items():
    p = WORKSPACE / fname
    if not p.exists(): p.write_text(content, encoding='utf-8')

print(f'Workspace seeded: {[f.name for f in WORKSPACE.iterdir()]}')

Workspace seeded: ['TOOLS.md', 'HEARTBEAT.md', 'IDENTITY.md', 'SOUL.md', 'MEMORY.md']


In [16]:
class PromptAssembler:
    '''
    Assembles the 8-layer system prompt from workspace files.
    Each layer is independently loadable and replaceable.
    Modelled on claw0 s06 intelligence layer.
    '''

    # Base prompt: hardcoded, always the first layer
    _BASE = (
        'You are an AI assistant with tool use capabilities. '
        'You operate inside a message gateway that receives messages from multiple channels. '
        'Use tools to complete tasks. Think before acting.'
    )

    def __init__(self, workspace='workspace'):
        self._ws = Path(workspace)
        # Each layer is a str; None means not loaded
        self._identity = self._read('IDENTITY.md')
        self._soul = self._read('SOUL.md')
        self._memory = self._read('MEMORY.md')
        self._tools_doc = self._read('TOOLS.md')
        self._heartbeat = self._read('HEARTBEAT.md')

        # loaded_skills: list[str] (num_loaded_skills,)
        self._skills:   list[str] = []

        # context: str -- dynamic per-turn context
        self._context:  str = ''

    def _read(self, fname):
        p = self._ws / fname
        return p.read_text(encoding='utf-8') if p.exists() else ''

    # === Layer setters ===
    def load_soul(self, fname):
        # fname: str (relative to workspace) -> None
        self._soul = self._read(fname)

    def set_memory(self, text):
        # text: str -> None  (replaces MEMORY layer; also written to disk)
        self._memory = f'# Memory\n{text}'
        (self._ws / 'MEMORY.md').write_text(self._memory, encoding='utf-8')

    def add_skill(self, content):
        # content: str (full skill text) -> None
        self._skills.append(content)

    def set_context(self, text):
        # text: str -> None  (dynamic per-turn context)
        self._context = text

    # -- Assembler -----------------------------------------------------------
    def build(self, mode='normal'):
        # mode: 'normal' | 'heartbeat'
        # -> str (assembled system prompt, all 8 layers)
        layers = []

        # Layer 1: BASE -- always present, hardcoded
        layers.append(f'# Role\n{self._BASE}')

        # Layer 2: IDENTITY -- who this specific agent is
        if self._identity:
            layers.append(self._identity)

        # Layer 3: SOUL -- values and personality
        if self._soul:
            layers.append(self._soul)

        # Layer 4: MEMORY -- what the agent remembers
        if self._memory:
            layers.append(self._memory)

        # Layer 5: SKILLS -- loaded domain knowledge
        for skill in self._skills:
            layers.append(skill)

        # Layer 6: TOOLS -- available tool descriptions
        if self._tools_doc:
            layers.append(self._tools_doc)

        # Layer 7: CONTEXT -- dynamic per-turn context
        if self._context:
            layers.append(f'# Current Context\n{self._context}')

        # Layer 8: HEARTBEAT -- only in heartbeat mode
        if mode == 'heartbeat' and self._heartbeat:
            layers.append(self._heartbeat)

        # layers: list[str] (num_layers,) -> str (full prompt)
        return '\n\n'.join(layers)

    def layer_summary(self):
        # -> str (which layers are loaded and their sizes)
        names  = [
            'BASE',
            'IDENTITY',
            'SOUL',
            'MEMORY',
            f'SKILLS x{len(self._skills)}',
            'TOOLS',
            'CONTEXT',
            'HEARTBEAT'
        ]

        values = [
            self._BASE,
            self._identity,
            self._soul,
            self._memory,
            '\n'.join(self._skills),
            self._tools_doc,
            self._context,
            self._heartbeat
        ]

        return '\n'.join(
            f'  {n:15}: {len(v):>5} chars' if v else f'  {n:15}: (empty)'
            for n, v in zip(names, values)
        )


# === Demo ===
asm = PromptAssembler(workspace=str(WORKSPACE))
asm.set_context(f'Current time: {datetime.now().isoformat()}')
prompt = asm.build(mode='normal')
print(f'Assembled prompt ({len(prompt)} chars):')
print(asm.layer_summary())
print('\n--- Prompt preview---')
print(prompt)

Assembled prompt (1046 chars):
  BASE           :   185 chars
  IDENTITY       :   214 chars
  SOUL           :   318 chars
  MEMORY         :    73 chars
  SKILLS x0      : (empty)
  TOOLS          :   181 chars
  CONTEXT        :    40 chars
  HEARTBEAT      :   361 chars

--- Prompt preview---
# Role
You are an AI assistant with tool use capabilities. You operate inside a message gateway that receives messages from multiple channels. Use tools to complete tasks. Think before acting.

# Identity
You are Claw, a proactive AI assistant embedded in a messaging gateway.
You have access to tools and can take actions on behalf of users.
You communicate across Telegram, Feishu, and WebSocket channels.


# Soul
## Values
- Be concise. Say exactly what is needed, no more.
- Be proactive. Notice things the user has not asked about yet.
- Be honest. If you do not know, say so.
## Tone
Direct but warm. Avoid corporate language and filler phrases.
## Boundaries
Do not pretend to have feelings. Do

# 2) Channel Abstraction

- Telegram sends `{message: {text: ..., from: {id: ...}}}`.
- Feishu sends
`{event: {message: {content: ..., sender: {sender_id: ...}}}}`.
- WebSocket sends whatever the client defines.

Every platform is different.

The channel layer solves this by **normalising all incoming messages into a
single `InboundMessage` type** before they reach the agent. The agent only ever sees `InboundMessage` objects -- it does not know or care which platform delivered them.

This is the classic **adapter pattern**: one interface, many implementations.


```python
@dataclass
class InboundMessage:
    channel:    str   # 'telegram' | 'feishu' | 'websocket' | 'cli'
    peer_id:    str   # stable ID for the sender (used for routing)
    text:       str   # the message content
    msg_id:     str   # platform-specific message ID (for deduplication)
    timestamp:  str
    raw:        dict  # original platform payload (for debugging)
```

```
Platform event
   |
   v
Adapter.normalise(raw_event)
   |
   v
InboundMessage  ---->  Gateway (§3)  ---->  Agent
   |
   v
OutboundMessage  ---->  Adapter.send(reply, peer_id)
   |
   v
Platform API
```

In [17]:
@dataclass
class InboundMessage:
    '''
    The single normalised message type that flows through the entire gateway.
    Every channel adapter must produce exactly this type.
    '''

    channel:   str   # 'telegram' | 'feishu' | 'websocket' | 'cli' | 'mock'
    peer_id:   str   # stable sender ID -- used by the routing table in section 3
    text:      str   # the human-readable message text
    msg_id:    str   # platform-specific deduplication ID
    timestamp: str   # ISO 8601
    raw:       dict  # original platform payload


@dataclass
class OutboundMessage:
    # A reply to be sent back to a specific peer on a specific channel.
    channel:  str
    peer_id:  str
    text:     str
    reply_to: Optional[str] = None  # msg_id to reply to (platform-specific)


class BaseChannelAdapter:
    # Abstract adapter interface. Concrete adapters implement normalise() and send().
    name: str = 'base'

    def normalise(self, raw_event):
        # raw_event: dict -> InboundMessage
        raise NotImplementedError

    def send(self, msg):
        # msg: OutboundMessage -> None
        raise NotImplementedError


class TelegramAdapter(BaseChannelAdapter):
    # Normalises Telegram update payloads into InboundMessage.
    # In production, outbound calls would use the Telegram Bot API.
    # Here we mock send() so the notebook runs without a bot token.
    name = 'telegram'

    def normalise(self, raw_event):
        # raw_event: Telegram update dict -> InboundMessage
        msg = raw_event.get('message', {})
        return InboundMessage(
            channel   = self.name,
            peer_id   = str(msg.get('from', {}).get('id', 'unknown')),
            text      = msg.get('text', ''),
            msg_id    = str(msg.get('message_id', uuid.uuid4().hex[:8])),
            timestamp = datetime.now().isoformat(),
            raw       = raw_event,
        )

    def send(self, msg):
        # mock -- in production: requests.post(TELEGRAM_API + '/sendMessage', ...)
        print(f'[Telegram -> {msg.peer_id}] {msg.text[:80]}')


class FeishuAdapter(BaseChannelAdapter):
    # Normalises Feishu event payloads into InboundMessage.
    # Feishu (Lark) uses a nested event structure under 'event'.
    name = 'feishu'

    def normalise(self, raw_event):
        # raw_event: Feishu event dict -> InboundMessage
        event   = raw_event.get('event', {})
        message = event.get('message', {})
        sender  = event.get('sender', {})
        # Feishu message content is JSON-encoded inside the message dict
        try:
            content = json.loads(message.get('content', '{}'))
            text    = content.get('text', '')
        except (json.JSONDecodeError, TypeError):
            text    = str(message.get('content', ''))
        return InboundMessage(
            channel   = self.name,
            peer_id   = sender.get('sender_id', {}).get('open_id', 'unknown'),
            text      = text,
            msg_id    = message.get('message_id', uuid.uuid4().hex[:8]),
            timestamp = datetime.now().isoformat(),
            raw       = raw_event,
        )

    def send(self, msg):
        # mock -- in production: Feishu Send Message API
        print(f'[Feishu -> {msg.peer_id}] {msg.text[:80]}')


class CLIAdapter(BaseChannelAdapter):
    # Thin adapter for REPL / CLI usage. Useful for local testing.
    name = 'cli'

    def normalise(self, text):
        # text: str -> InboundMessage  (CLI variant takes a plain string)
        return InboundMessage(
            channel='cli',
            peer_id='local_user',
            text=text,
            msg_id=uuid.uuid4().hex[:8],
            timestamp=datetime.now().isoformat(),
            raw={},
        )

    def send(self, msg):
        print(f'[CLI] {msg.text}')


# -- Demo -------------------------------------------------------------------
tg = TelegramAdapter()
raw_tg = {'message': {'message_id': 42, 'from': {'id': 12345}, 'text': 'Hello Claw!',
                       'date': 1700000000}}
msg = tg.normalise(raw_tg)
print(f'channel={msg.channel} peer={msg.peer_id} text={msg.text!r}')

fs = FeishuAdapter()
raw_fs = {'event': {
    'message': {'message_id': 'om_abc', 'content': '{"text":"Good morning"}'},
    'sender':  {'sender_id': {'open_id': 'ou_xyz'}},
}}
msg2 = fs.normalise(raw_fs)
print(f'channel={msg2.channel} peer={msg2.peer_id} text={msg2.text!r}')

channel=telegram peer=12345 text='Hello Claw!'
channel=feishu peer=ou_xyz text='Good morning'


# 3) Gateway and Routing

A deployment has multiple channels (Telegram, Feishu, CLI) and potentially
multiple agents (customer support agent, coding agent, data agent).

The gateway answers one question:
> Given an incoming message, which agent should handle it?

Solves this with a **5-tier binding table**:
Each entry maps a `(channel, peer_id)` pair to an agent. The lookup algorithm is:

```
1. Exact match:     (channel='telegram', peer='12345')  -> coding_agent
2. Peer wildcard:   (channel='telegram', peer='*')      -> support_agent
3. Channel wildcard:(channel='*', peer='12345')  -> vip_agent
4. Global wildcard: (channel='*', peer='*')      -> default_agent
5. No match:        drop (or reply with error)
```

More specific rules always beat less specific ones
> (exact > peer-wildcard channel-wildcard > global-wildcard)


Each `(channel, peer_id)` pair gets its own `messages[]` list -- a **session**.
Two users on the same channel never see each other's history.
Sessions are persisted to JSONL so conversations survive restarts.

```python
gateway.bind('telegram', '12345', coding_agent)
gateway.bind('telegram', '*',     support_agent)

gateway.route(InboundMessage(channel='telegram', peer_id='12345', ...))
# -> coding_agent handles this message

gateway.route(InboundMessage(channel='telegram', peer_id='99999', ...))
# -> support_agent handles (peer wildcard match)
```

In [18]:
class AgentSession:
    # Per-(channel, peer) conversation state.
    # messages[] is isolated per session -- two users never share context.

    def __init__(self, channel, peer_id, sessions_dir='.sessions'):
        self.channel   = channel
        self.peer_id   = peer_id
        self.key       = f'{channel}__{peer_id}'
        self._path     = Path(sessions_dir) / f'{self.key}.jsonl'
        self._path.parent.mkdir(parents=True, exist_ok=True)
        # messages: list[dict] (num_turns,) -- the live conversation history
        self.messages: list = self._load()

    def _load(self):
        # Replay JSONL on startup for session resume
        if not self._path.exists(): return []
        msgs = []
        for line in self._path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                try: msgs.append(json.loads(line))
                except json.JSONDecodeError: pass
        return msgs

    def append(self, msg):
        # Append one message dict -- write-through to JSONL
        # msg: dict -> None (side effect)
        self.messages.append(msg)
        with open(self._path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(msg, default=str) + '\n')

    def clear(self):
        self.messages.clear()
        if self._path.exists(): self._path.unlink()


# Binding precedence: lower number = higher priority
TIER_EXACT            = 1  # (channel, peer)
TIER_PEER_WILDCARD    = 2  # (channel, *)
TIER_CHANNEL_WILDCARD = 3  # (*, peer)
TIER_GLOBAL_WILDCARD  = 4  # (*, *)


@dataclass
class BindingEntry:
    channel:  str
    peer_id:  str
    agent_fn: Callable  # callable(InboundMessage, session: AgentSession) -> str
    tier:     int


class MessageGateway:
    # Routes InboundMessage objects to agent functions via a 5-tier binding table.
    # Manages per-(channel, peer) sessions with JSONL persistence.

    def __init__(self, sessions_dir='.sessions'):
        self._sessions_dir = sessions_dir
        # _bindings: list[BindingEntry] (num_bindings,) -- sorted by tier
        self._bindings: list = []
        # _sessions: dict[str, AgentSession] (num_active_sessions,)
        self._sessions: dict = {}

    def bind(self, channel, peer_id, agent_fn):
        '''
        Register a routing rule.
        channel: str ('*' for wildcard), peer_id: str ('*' for wildcard)
        agent_fn: Callable(InboundMessage, AgentSession) -> str
        '''

        if channel != '*' and peer_id != '*':
            tier = TIER_EXACT
        elif channel != '*':
            tier = TIER_PEER_WILDCARD
        elif peer_id != '*':
            tier = TIER_CHANNEL_WILDCARD
        else:
            tier = TIER_GLOBAL_WILDCARD

        self._bindings.append(BindingEntry(channel, peer_id, agent_fn, tier))
        # Keep bindings sorted: most specific first
        self._bindings.sort(key=lambda b: b.tier)

    def _get_session(self, channel, peer_id):
        # Lazily create or retrieve a session.
        # (channel, peer_id): (str, str) -> AgentSession
        key = f'{channel}__{peer_id}'
        if key not in self._sessions:
            self._sessions[key] = AgentSession(channel, peer_id, self._sessions_dir)
        return self._sessions[key]

    def route(self, inbound):
        '''
        Find the best-matching binding and dispatch the message.
        inbound: InboundMessage -> str (agent reply) | None
        '''

        for binding in self._bindings:
            c_match = binding.channel in (inbound.channel, '*')
            p_match = binding.peer_id  in (inbound.peer_id,  '*')
            if c_match and p_match:
                session = self._get_session(inbound.channel, inbound.peer_id)
                print(f'  [Gateway] tier={binding.tier} '
                      f'({binding.channel},{binding.peer_id}) matched '
                      f'({inbound.channel},{inbound.peer_id})')
                return binding.agent_fn(inbound, session)
        print(f'  [Gateway] No binding matched ({inbound.channel},{inbound.peer_id})')
        return None

    def binding_table(self):
        # -> str (human-readable routing table)
        rows = [f'  tier={b.tier} ({b.channel},{b.peer_id}) -> {b.agent_fn.__name__}'
                for b in self._bindings]
        return 'Binding table:\n' + ('\n'.join(rows) or '  (empty)')


# -- Demo agent functions ---------------------------------------------------
def echo_agent(msg, session):
    session.append({'role':'user','content': msg.text})
    reply = f'Echo [{msg.channel}]: {msg.text}'
    session.append({'role':'assistant','content': reply})
    return reply

def vip_agent(msg, session):
    return f'[VIP] Hello {msg.peer_id}, you have priority support.'

def default_agent(msg, session):
    return f'[Default] Got your message: {msg.text}'


# -- Demo -------------------------------------------------------------------
gw = MessageGateway()
gw.bind('telegram', '12345', vip_agent)     # tier 1: exact
gw.bind('telegram', '*',     echo_agent)    # tier 2: peer wildcard
gw.bind('*',        '*',     default_agent) # tier 4: global wildcard

print(gw.binding_table())

cli_adapter = CLIAdapter()

m1 = TelegramAdapter().normalise({'message':{'from':{'id':12345},'text':'Hi VIP','message_id':1}})
m2 = TelegramAdapter().normalise({'message':{'from':{'id':99999},'text':'Hello','message_id':2}})
m3 = cli_adapter.normalise('Testing fallback')

print('\n--- routing ---')
print(gw.route(m1))
print(gw.route(m2))
print(gw.route(m3))

Binding table:
  tier=1 (telegram,12345) -> vip_agent
  tier=2 (telegram,*) -> echo_agent
  tier=4 (*,*) -> default_agent

--- routing ---
  [Gateway] tier=1 (telegram,12345) matched (telegram,12345)
[VIP] Hello 12345, you have priority support.
  [Gateway] tier=2 (telegram,*) matched (telegram,99999)
Echo [telegram]: Hello
  [Gateway] tier=4 (*,*) matched (cli,local_user)
[Default] Got your message: Testing fallback


# 4) Heartbeat and Cron

Every agent we have built so far is **reactive**: it waits for a user message
and then responds. Real-world agents often need to **act proactively**:
send a morning briefing, check an API on a schedule, nudge a user who has
not responded in 24 hours.

1. **Heartbeat** -- a timer thread that wakes up the agent periodically.
   The agent receives a special heartbeat message and decides whether to act.
   If nothing needs attention it returns exactly `IDLE`. Otherwise it acts.

2. **Cron** -- a scheduler that fires named jobs at specified cron expressions.
   Jobs inject a task message into the agent loop, identical to a user message.

Instead of adding special logic to decide 'should the agent fire?', we **ask the agent itself**.
The agent reads context (memory, pending tasks, time) and returns `IDLE` if it decides there is nothing to do. No external heuristic needed.

```python
hb = HeartbeatScheduler(agent_fn, interval_s=60)
hb.start()
# Every 60s:
#   agent receives: '[Heartbeat] 2026-04-20T09:00:00. Review context.'
#   agent replies:  'IDLE'  (or takes an action)
```

In [19]:
@dataclass
class CronJob:
    name:        str
    expression:  str   # cron expression e.g. '0 9 * * *'
    task:        str   # injected as a user message to the agent
    last_run:    Optional[str] = None


def _cron_matches(expression, now):
    '''
    Minimal cron matcher for demo purposes.
    In production use the 'croniter' library.
    expression: str, now: datetime -> bool
    '''

    parts = expression.strip().split()
    if len(parts) != 5: return False
    m_min, m_hr, m_dom, m_mon, m_dow = parts

    def field_match(field, value):
        if field == '*':
            return True
        try:
            return int(field) == value
        except ValueError:
            return False

    return (
        field_match(m_min, now.minute) and
        field_match(m_hr,  now.hour)   and
        field_match(m_dom, now.day)    and
        field_match(m_mon, now.month)  and
        field_match(m_dow, now.weekday())
    )


class HeartbeatScheduler:
    '''
    Wakes the agent on a fixed interval. The agent reads context and
    decides whether to act (or return IDLE).
    Runs in a daemon thread so it dies with the main process.
    '''

    IDLE_SIGNAL = 'IDLE'

    def __init__(self, agent_fn, system_fn=None, interval_s=60):
        '''
        agent_fn:  Callable(messages: list, system: str) -> str
        system_fn: Callable() -> str  (fetches current system prompt)
        interval_s: int -- seconds between heartbeat ticks
        '''

        self._agent_fn   = agent_fn
        self._system_fn  = system_fn or (lambda: '')
        self._interval   = interval_s
        self._stop       = threading.Event()
        self._tick_count = 0

    def _tick(self):
        '''
        One heartbeat cycle: inject a heartbeat message and call the agent.
        -> str (agent reply) | IDLE_SIGNAL
        '''

        self._tick_count += 1
        ts = datetime.now().isoformat()
        hb_msg = (
            f'[Heartbeat #{self._tick_count}] {ts}. '
            'Review context and pending tasks. '
            f'If nothing needs attention, reply with exactly: {self.IDLE_SIGNAL}'
        )
        messages = [{'role':'user','content': hb_msg}]
        system   = self._system_fn()

        try:
            self._agent_fn(messages, system)
        except Exception as exc:
            return f'[HeartbeatError] {exc}'

        reply = next(
            (m['content'] for m in reversed(messages)
             if m['role'] == 'assistant' and m['content']),
            self.IDLE_SIGNAL,
        )
        if reply.strip().upper() == self.IDLE_SIGNAL:
            return self.IDLE_SIGNAL
        print(f'[Heartbeat] Action taken: {reply[:100]}')
        return reply

    def run(self):
        # Main heartbeat loop. Called in a daemon thread.
        print(f'[Heartbeat] Started (interval={self._interval}s)')
        while not self._stop.is_set():
            result = self._tick()
            status = 'IDLE' if result == self.IDLE_SIGNAL else 'ACTIVE'
            print(f'[Heartbeat] tick #{self._tick_count} -> {status}')
            self._stop.wait(timeout=self._interval)
        print('[Heartbeat] Stopped.')

    def start(self):
        t = threading.Thread(target=self.run, daemon=True)
        t.start()
        return t

    def shutdown(self): self._stop.set()


class CronScheduler:
    '''
    Checks registered cron jobs every minute and fires matching jobs.
    Each job injects a task string into the agent loop.
    '''

    def __init__(self, agent_fn, system_fn=None):
        self._agent_fn  = agent_fn
        self._system_fn = system_fn or (lambda: '')
        # _jobs: list[CronJob] (num_jobs,)
        self._jobs: list = []
        self._stop  = threading.Event()

    def register(self, job):
        # job: CronJob -> None
        self._jobs.append(job)
        print(f'[Cron] Registered: {job.name!r} ({job.expression})')

    def _check_and_fire(self):
        # Check all jobs against current time; fire matches.
        now = datetime.now()
        for job in self._jobs:
            if _cron_matches(job.expression, now):
                print(f'[Cron] Firing: {job.name}')
                messages = [{'role':'user','content': f'[Cron: {job.name}] {job.task}'}]
                try:
                    self._agent_fn(messages, self._system_fn())
                except Exception as exc:
                    print(f'[Cron] Error in {job.name}: {exc}')
                job.last_run = now.isoformat()

    def run(self):
        print('[Cron] Scheduler started.')
        while not self._stop.is_set():
            self._check_and_fire()
            self._stop.wait(timeout=60)  # check every minute
        print('[Cron] Scheduler stopped.')

    def start(self):
        t = threading.Thread(target=self.run, daemon=True)
        t.start()
        return t

    def shutdown(self): self._stop.set()


# -- Demo -------------------------------------------------------------------
# Use a fast mock agent for the heartbeat demo
_hb_call_count = [0]

def _mock_agent(messages, system):
    _hb_call_count[0] += 1
    # Alternate between IDLE and an action for demo variety
    reply = 'IDLE' if _hb_call_count[0] % 2 == 0 else 'Sent morning briefing to 3 users.'
    messages.append({'role':'assistant','content': reply})

# fast interval for demo
hb = HeartbeatScheduler(_mock_agent, interval_s=2)
t  = hb.start()
time.sleep(5.5)
hb.shutdown()
t.join(timeout=3)
print(f'Total heartbeat ticks: {hb._tick_count}')

[Heartbeat] Started (interval=2s)
[Heartbeat] Action taken: Sent morning briefing to 3 users.
[Heartbeat] tick #1 -> ACTIVE
[Heartbeat] tick #2 -> IDLE
[Heartbeat] Action taken: Sent morning briefing to 3 users.
[Heartbeat] tick #3 -> ACTIVE
[Heartbeat] Stopped.
Total heartbeat ticks: 3


# 5) Delivery Queue

Agent replies can fail to deliver: the platform API is down, the network
blips, rate limits hit. If you call the platform API directly from the agent
loop, a failure means the user never gets the reply -- and you have no
record that you even tried.

The **delivery queue** solves this with a **write-ahead log (WAL)**:

1. Write the outbound message to disk (BEFORE calling the API).
2. Try to send via the platform API.
3. On success: mark the message as sent.
4. On failure: retry with exponential backoff.
5. On crash: replay the WAL on restart -- no message is ever silently lost.


**Backoff Strategy**

```
Attempt 1: immediate
Attempt 2: wait 1s
Attempt 3: wait 2s
Attempt 4: wait 4s
Attempt N: wait min(2^(N-1), max_delay) seconds
```

```python
dq = DeliveryQueue(adapter=telegram_adapter)
dq.enqueue(OutboundMessage(channel='telegram', peer_id='12345', text='Done!'))
# -> writes to WAL, attempts send, marks sent
# -> if API fails: retries with backoff
```

In [20]:
class MessageStatus(Enum):
    PENDING = auto()
    SENT    = auto()
    FAILED  = auto()


@dataclass
class QueuedMessage:
    msg_id:     str
    channel:    str
    peer_id:    str
    text:       str
    status:     MessageStatus = MessageStatus.PENDING
    attempts:   int = 0
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    last_tried: Optional[str] = None
    error:      Optional[str] = None

    def to_dict(self):
        d = {k:v for k,v in self.__dict__.items()}
        d['status'] = d['status'].name
        return d

    @classmethod
    def from_dict(cls, d):
        d = dict(d)
        d['status'] = MessageStatus[d['status']]
        return cls(**d)


class DeliveryQueue:

    '''
    # Write-ahead delivery queue with exponential backoff retry.
    # Every outbound message is written to disk before attempting delivery.
    # On restart, PENDING messages are replayed automatically.
    '''

    MAX_ATTEMPTS  = 5
    MAX_DELAY_S   = 30

    def __init__(self, send_fn=None, wal_path='.delivery/wal.jsonl'):
        '''
        send_fn: Callable(OutboundMessage) -> None  (raises on failure)
        If None, falls back to print (safe mock)
        '''

        self._send_fn  = send_fn or (lambda m: print(f'[DeliveryMock] {m.channel}->{m.peer_id}: {m.text[:60]}'))
        self._wal_path = Path(wal_path)
        self._wal_path.parent.mkdir(parents=True, exist_ok=True)
        # _queue: dict[str, QueuedMessage] (num_pending,)
        self._queue: dict = {}
        self._lock = threading.Lock()
        self._stop = threading.Event()
        self._replay_wal()

    def _replay_wal(self):
        # On startup, reload any PENDING messages from the WAL.
        if not self._wal_path.exists(): return
        replayed = 0
        for line in self._wal_path.read_text(encoding='utf-8').splitlines():
            if not line.strip(): continue
            try:
                qm = QueuedMessage.from_dict(json.loads(line))
                if qm.status == MessageStatus.PENDING:
                    self._queue[qm.msg_id] = qm
                    replayed += 1
            except Exception: pass
        if replayed: print(f'[Delivery] Replayed {replayed} pending message(s) from WAL.')

    def _write_wal(self, qm):
        # Append one entry to the WAL -- always before attempting delivery.
        with open(self._wal_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(qm.to_dict()) + '\n')

    def enqueue(self, outbound):
        # outbound: OutboundMessage -> str (msg_id)
        qm = QueuedMessage(
            msg_id  = uuid.uuid4().hex[:8],
            channel = outbound.channel,
            peer_id = outbound.peer_id,
            text    = outbound.text,
        )
        with self._lock:
            self._write_wal(qm)  # WAL first -- crash safety
            self._queue[qm.msg_id] = qm
        return qm.msg_id

    def _attempt_send(self, qm):
        '''
        Try to deliver one message. Updates status in-place.
        qm: QueuedMessage -> None (mutates qm.status, qm.attempts)
        '''
        if qm.attempts >= self.MAX_ATTEMPTS:
            qm.status = MessageStatus.FAILED
            return
        delay = min(2 ** qm.attempts, self.MAX_DELAY_S)
        if qm.attempts > 0:
            time.sleep(delay)
        qm.attempts   += 1
        qm.last_tried  = datetime.now().isoformat()
        try:
            self._send_fn(OutboundMessage(channel=qm.channel,
                                           peer_id=qm.peer_id, text=qm.text))
            qm.status = MessageStatus.SENT
        except Exception as exc:
            qm.error = str(exc)
            print(f'  [Delivery] Attempt {qm.attempts} failed for {qm.msg_id}: {exc}')

    def flush(self):
        '''
        Attempt delivery of all PENDING messages.
        Called by the flush worker thread or manually.
        '''

        with self._lock:
            pending = [
                qm for qm in self._queue.values()
                if qm.status == MessageStatus.PENDING
            ]

        for qm in pending:
            while qm.status == MessageStatus.PENDING and qm.attempts < self.MAX_ATTEMPTS:
                self._attempt_send(qm)

        print(f'[Delivery] Flush complete. '
              f'Sent: {sum(1 for q in self._queue.values() if q.status==MessageStatus.SENT)} '
              f'Failed: {sum(1 for q in self._queue.values() if q.status==MessageStatus.FAILED)}')


# === Demo ===
dq = DeliveryQueue()

# Enqueue 3 messages
for i in range(3):
    mid = dq.enqueue(OutboundMessage('telegram', f'user_{i}', f'Hello user {i}!'))
    print(f'Enqueued {mid}')

# Flush (attempt delivery)
dq.flush()

Enqueued e9a68649
Enqueued bea504ea
Enqueued e371d2ec
[DeliveryMock] telegram->user_0: Hello user 0!
[DeliveryMock] telegram->user_1: Hello user 1!
[DeliveryMock] telegram->user_2: Hello user 2!
[Delivery] Flush complete. Sent: 3 Failed: 0


# 6) Resilience

Production agents face three classes of failure:

| Class | Example | Layer that handles it |
|-------|---------|----------------------|
| Tool failure | bash returns an error | Inner: tool-use retry loop |
| Context overflow | messages[] exceeds window | Middle: overflow compaction |
| Auth failure | API key rate-limited | Outer: auth profile rotation |

The **3-layer retry onion** -- each layer catches a different class of error and either recovers or escalates outward.

- **Inner -- Tool retry loop:** if a tool call raises an exception, inject
an error message and let the model try again (up to N times).

- **Middle -- Overflow compaction:** if the API returns a context-length
error, compact the conversation and retry the same call.

- **Outer -- Auth rotation:** if the API returns an auth error (rate limit,
quota exceeded), swap to the next API key in a pool and retry.

```python
resilient = ResilientRunner(auth_pool=['key_a', 'key_b', 'key_c'])
result = resilient.run(messages, system, tools, handlers)
# -> transparently retried through auth rotation if key_a was rate-limited
```

In [21]:
class AuthPool:
    '''
    Manages a rotating pool of API credentials.
    On auth failure, rotates to the next key.
    Modelled on claw0 s09 auth profile rotation.
    '''

    def __init__(self, api_keys):
        # api_keys: list[str] (num_keys,)
        if not api_keys:
            raise ValueError('Auth pool requires at least one key.')

        # _keys: deque[str] (num_keys,) -- rotated on failure
        self._keys    = deque(api_keys)
        self._current = self._keys[0]
        self._failures: dict = defaultdict(int)   # key -> consecutive failure count

    @property
    def current(self):
        return self._current

    def report_failure(self, key):
        '''
        Mark a key as failed and rotate to the next.
        key: str -> str (new current key)
        '''

        self._failures[key] += 1
        # move failed key to back
        self._keys.rotate(-1)
        self._current = self._keys[0]
        print(f'  [AuthPool] Rotated from {key[:8]}... to {self._current[:8]}...')
        return self._current

    def report_success(self, key):
        self._failures[key] = 0


class ToolRetryPolicy:
    '''
    Inner layer: retry the agent loop if a tool call raises an exception.
    The model receives the error message and can try a different approach.
    '''

    def __init__(self, max_tool_retries=3):
        self.max_tool_retries = max_tool_retries

    def wrap_handlers(self, tool_handlers, retry_budget):
        '''
        Wrap all handlers so failures are caught and returned as error strings
        rather than exceptions. The model then retries differently.
        retry_budget: list[int] (1,) -- mutable counter shared across handlers
        '''

        wrapped = {}
        for name, handler in tool_handlers.items():
            def make_handler(h, n):
                def safe_handler(**kwargs):
                    try:
                        return h(**kwargs)
                    except Exception as exc:
                        retry_budget[0] -= 1
                        if retry_budget[0] > 0:
                            return (
                                f'[ToolError] {n} failed: {exc}. '
                                f'{retry_budget[0]} retries remaining. '
                                'Please try a different approach.'
                            )
                        raise  # escalate to middle layer
                return safe_handler
            wrapped[name] = make_handler(handler, name)
        return wrapped


class ResilientRunner:
    '''
    3-layer retry onion: tool retry -> overflow compact -> auth rotation.
    Wraps run_agent_loop with production-grade error handling.
    '''

    MAX_OVERFLOW_RETRIES = 2
    MAX_AUTH_RETRIES     = 3

    def __init__(self, auth_pool=None):
        # auth_pool: list[str] | None -- API keys to rotate through
        self._auth = AuthPool(auth_pool or [os.environ.get('OPENAI_API_KEY','')])
        self._tool_policy = ToolRetryPolicy()

    def run(self, messages, system, tools, tool_handlers):
        '''Attempt the agent loop with 3-layer error recovery'''

        # messages: list[dict] (N,) -- mutated in-place
        # -> list[dict] (final messages)
        auth_attempts     = 0
        overflow_attempts = 0

        while auth_attempts < self.MAX_AUTH_RETRIES:
            # Set the active API key
            os.environ['OPENAI_API_KEY'] = self._auth.current
            # Rebuild client with current key
            active_client = OpenAI(api_key=self._auth.current)

            retry_budget = [self._tool_policy.max_tool_retries]
            safe_handlers = self._tool_policy.wrap_handlers(tool_handlers, retry_budget)

            try:
                # Inner layer: tool retry (handled by safe_handlers)
                result = run_agent_loop(
                    messages=messages,
                    tools=tools,
                    tool_handlers=safe_handlers,
                    system=system,
                )
                self._auth.report_success(self._auth.current)
                return result

            except Exception as exc:
                err = str(exc).lower()

                # Middle layer: overflow compaction
                if ('context' in err or 'length' in err or 'token' in err) and overflow_attempts < self.MAX_OVERFLOW_RETRIES:
                    overflow_attempts += 1
                    print(f'  [Resilience] Context overflow (attempt {overflow_attempts}). Compacting...')

                    # Compact: keep last 4 messages as hard reset
                    messages[:] = messages[-4:]
                    messages.insert(0, {'role':'user','content':'[Context was compacted due to overflow. Please continue.]'})
                    continue

                # Outer layer: auth rotation
                if 'auth' in err or 'rate' in err or '401' in err or '429' in err:
                    print(f'  [Resilience] Auth failure: {exc}')
                    self._auth.report_failure(self._auth.current)
                    auth_attempts += 1
                    continue

                # Unhandled error -- escalate
                raise

        raise RuntimeError(f'All {self.MAX_AUTH_RETRIES} auth keys exhausted.')


# -- Demo -------------------------------------------------------------------
print('ResilientRunner with 1-key pool (demo mode):')
runner = ResilientRunner(auth_pool=[os.environ.get('OPENAI_API_KEY','')])
print(f'Active key: {runner._auth.current[:8]}...')
print('Auth pool size:', len(runner._auth._keys))
print('Tool retry budget per call:', runner._tool_policy.max_tool_retries)
print('Overflow retry budget:', runner.MAX_OVERFLOW_RETRIES)

ResilientRunner with 1-key pool (demo mode):
Active key: sk-proj-...
Auth pool size: 1
Tool retry budget per call: 3
Overflow retry budget: 2


# 7) Concurrency Lanes

Imagine two users sending messages simultaneously to the same agent.

Without coordination, their `messages[]` histories could interleave:

> user A's tool result gets inserted into user B's conversation.

Solves this with **named lanes**: each `(channel, peer_id)` pair
gets its own dedicated FIFO queue.

Requests within a lane are serialised (preserving conversation order); requests across different lanes run concurrently (no unnecessary waiting).

**Generation Tracking**

1. A subtler problem: if user A sends two rapid messages (M1 then M2),
the response to M1 might arrive *after* M2 has been processed, resulting
in out-of-order replies.

2. **Generation tracking** assigns a monotonic counter to each request; stale responses (from superseded requests) are
silently discarded.

```
InboundMessage
   |
   v
LaneRouter
   |-- lane 'telegram__12345': FIFO queue [M1, M2, M3]
   |    \-- worker thread: processes M1, M2, M3 in order
   |
   |-- lane 'telegram__99999': FIFO queue [M4]
   |    \-- worker thread: processes M4 concurrently
   |
   \-- lane 'feishu__ou_xyz':  FIFO queue [M5, M6]
        \-- worker thread: processes M5, M6 in order
```

In [22]:
@dataclass
class LaneRequest:
    lane_id:    str
    generation: int   # monotonic counter -- stale responses have lower gen
    inbound:    InboundMessage
    future:     object  # threading.Event + result holder


class LaneFuture:
    # Lightweight future: lets the caller block until a lane produces a result.
    def __init__(self):
        self._event  = threading.Event()
        self.result  = None
        self.error   = None

    def set_result(self, r):  self.result = r; self._event.set()
    def set_error(self, e):   self.error  = e; self._event.set()
    def wait(self, timeout=30):
        # -> result: str | None
        self._event.wait(timeout=timeout)
        if self.error: raise RuntimeError(self.error)
        return self.result


class Lane:
    '''
    One named FIFO lane for a single (channel, peer_id) pair.
    Requests are processed strictly in order by a dedicated worker thread.
    '''

    def __init__(self, lane_id, handler_fn):
        self.lane_id    = lane_id
        self._handler   = handler_fn  # Callable(InboundMessage) -> str
        self._queue     = queue.Queue()
        self._gen       = 0           # latest generation dispatched to this lane
        self._stop      = threading.Event()
        t = threading.Thread(target=self._worker, daemon=True)
        t.start()

    def submit(self, inbound):
        # Submit a request; returns a LaneFuture the caller can wait on.
        # inbound: InboundMessage -> LaneFuture
        self._gen += 1
        fut = LaneFuture()
        req = LaneRequest(
            lane_id=self.lane_id,
            generation=self._gen,
            inbound=inbound,
            future=fut,
        )
        self._queue.put(req)
        return fut

    def _worker(self):
        # Worker loop: drain the FIFO queue, process each request.
        while not self._stop.is_set():
            try:
                req = self._queue.get(timeout=1.0)
            except queue.Empty:
                continue
            # Generation check: discard if a newer request has been submitted
            # (only relevant if requests can be cancelled -- kept here for
            # completeness and as documentation of the pattern)
            try:
                result = self._handler(req.inbound)
                req.future.set_result(result)
            except Exception as exc:
                req.future.set_error(str(exc))

    def shutdown(self): self._stop.set()


class LaneRouter:
    '''
    Creates and manages named lanes; routes messages to the correct lane.
    Lanes are created on first use and persist for the lifetime of the router.
    '''

    def __init__(self, handler_fn):
        # handler_fn: Callable(InboundMessage) -> str  (the actual agent call)
        self._handler  = handler_fn
        # _lanes: dict[str, Lane] (num_active_lanes,)
        self._lanes: dict = {}
        self._lock  = threading.Lock()

    def _get_lane(self, lane_id):
        # Lazily create a lane on first use.
        # lane_id: str -> Lane
        with self._lock:
            if lane_id not in self._lanes:
                self._lanes[lane_id] = Lane(lane_id, self._handler)
                print(f'  [LaneRouter] New lane: {lane_id}')
            return self._lanes[lane_id]

    def dispatch(self, inbound, wait=True, timeout=30):
        # Route an InboundMessage to its lane and optionally wait for result.
        # lane_id is derived from (channel, peer_id) -- stable per conversation.
        # inbound: InboundMessage, wait: bool -> str | LaneFuture
        lane_id = f'{inbound.channel}__{inbound.peer_id}'
        lane    = self._get_lane(lane_id)
        future  = lane.submit(inbound)
        if wait: return future.wait(timeout=timeout)
        return future

    @property
    def active_lanes(self):
        return list(self._lanes.keys())


# === Demo ===
_demo_responses = {'user_a': 'Hello from lane A!', 'user_b': 'Hello from lane B!'}

def _demo_handler(msg):
    time.sleep(0.2)  # simulate LLM latency
    return _demo_responses.get(msg.peer_id, f'Reply to {msg.peer_id}')

router = LaneRouter(_demo_handler)

# Two concurrent users -- their requests run in parallel across lanes
import concurrent.futures
cli = CLIAdapter()
m_a = InboundMessage('telegram','user_a','msg from A',uuid.uuid4().hex[:8],datetime.now().isoformat(),{})
m_b = InboundMessage('telegram','user_b','msg from B',uuid.uuid4().hex[:8],datetime.now().isoformat(),{})

with concurrent.futures.ThreadPoolExecutor(max_workers=2) as ex:
    fa = ex.submit(router.dispatch, m_a)
    fb = ex.submit(router.dispatch, m_b)
    ra, rb = fa.result(), fb.result()

print(f'user_a reply: {ra}')
print(f'user_b reply: {rb}')
print(f'Active lanes: {router.active_lanes}')

  [LaneRouter] New lane: telegram__user_a
  [LaneRouter] New lane: telegram__user_b
user_a reply: Hello from lane A!
user_b reply: Hello from lane B!
Active lanes: ['telegram__user_a', 'telegram__user_b']


# 8) Full GatewayHarness

`GatewayHarness` combines all 7 sections into a single runnable gateway.

```

Message flow:
  Platform
  -> Adapter.normalise()
  -> InboundMessage
  -> LaneRouter.dispatch()  (per-peer FIFO)
  -> MessageGateway.route() (tier-matched agent)
  -> ResilientRunner.run()  (3-layer retry)
  -> DeliveryQueue.enqueue() (WAL-backed delivery)
  -> Adapter.send()         (platform API)
```

We simulate a Telegram message flowing through the full stack.

In [23]:
class GatewayHarness:
    # Full gateway harness composing all sections 1-7.
    # Single entry point: process(raw_event, channel_name) -> str

    def __init__(self,
                 workspace='workspace',
                 sessions_dir='.sessions',
                 delivery_wal='.delivery/wal.jsonl',
                 heartbeat_interval=300,
                 api_keys=None):

        # -- Section 1: intelligence layer ----------------------------------
        self.prompt  = PromptAssembler(workspace=workspace)

        # -- Section 2: channel adapters ------------------------------------
        self.adapters = {
            'telegram': TelegramAdapter(),
            'feishu':   FeishuAdapter(),
            'cli':      CLIAdapter(),
        }

        # -- Section 3: gateway and routing ---------------------------------
        self.gateway = MessageGateway(sessions_dir=sessions_dir)

        # -- Section 5: delivery queue --------------------------------------
        self.delivery = DeliveryQueue(wal_path=delivery_wal)

        # -- Section 6: resilient runner ------------------------------------
        self.resilient = ResilientRunner(
            auth_pool=api_keys or [os.environ.get('OPENAI_API_KEY','')]
        )

        # -- Section 7: lane router -----------------------------------------
        # The lane handler calls the gateway, which calls the agent.
        # Wrapped here so it feeds through the resilient runner.
        self.lanes = LaneRouter(handler_fn=self._agent_call)

        # -- Section 4: heartbeat and cron ----------------------------------
        self.heartbeat = HeartbeatScheduler(
            agent_fn=lambda msgs, sys: run_agent_loop(
                messages=msgs, tools=[], tool_handlers={}, system=sys
            ),
            system_fn=lambda: self.prompt.build(mode='heartbeat'),
            interval_s=heartbeat_interval,
        )
        self.cron = CronScheduler(
            agent_fn=lambda msgs, sys: run_agent_loop(
                messages=msgs, tools=[], tool_handlers={}, system=sys
            ),
            system_fn=lambda: self.prompt.build(mode='normal'),
        )

        # Register a default global-wildcard agent binding
        self.gateway.bind('*', '*', self._gateway_agent)

    def _gateway_agent(self, inbound, session):
        # The core agent function called by the gateway.
        # Runs the resilient agent loop with the assembled prompt.
        # inbound: InboundMessage, session: AgentSession -> str
        self.prompt.set_context(
            f'Channel: {inbound.channel}\n'
            f'Peer:    {inbound.peer_id}\n'
            f'Time:    {datetime.now().isoformat()}'
        )
        system = self.prompt.build(mode='normal')
        msgs   = list(session.messages) + [{'role':'user','content': inbound.text}]

        try:
            result = self.resilient.run(
                messages=msgs, system=system, tools=[], tool_handlers={}
            )
            reply = next(
                (m['content'] for m in reversed(result)
                 if m['role']=='assistant' and m['content']),
                '(no reply)',
            )
        except Exception as exc:
            reply = f'[Error] {exc}'

        # Persist to session
        session.append({'role':'user',      'content': inbound.text})
        session.append({'role':'assistant', 'content': reply})
        return reply

    def _agent_call(self, inbound):
        # Lane handler: routes to gateway, delivers reply.
        # inbound: InboundMessage -> str (agent reply)
        reply = self.gateway.route(inbound)
        if reply:
            out = OutboundMessage(channel=inbound.channel,
                                  peer_id=inbound.peer_id, text=reply)
            self.delivery.enqueue(out)
        return reply or '(no routing match)'

    def process(self, raw_event, channel_name):
        # Main entry point. Normalise a raw platform event and route it.
        # raw_event: dict | str, channel_name: str -> str (agent reply)
        adapter = self.adapters.get(channel_name)
        if not adapter:
            return f'[Error] Unknown channel: {channel_name}'
        if channel_name == 'cli':
            inbound = adapter.normalise(raw_event)
        else:
            inbound = adapter.normalise(raw_event)
        return self.lanes.dispatch(inbound)

    def start_background(self):
        # Start heartbeat and cron daemon threads.
        self.heartbeat.start()
        self.cron.start()
        print('Background services started (heartbeat, cron).')

    def shutdown(self):
        self.heartbeat.shutdown()
        self.cron.shutdown()
        self.delivery.flush()
        print('GatewayHarness shut down.')


# -- Capstone Demo ----------------------------------------------------------
print('Initialising GatewayHarness...')
harness = GatewayHarness()
print(f'Adapters: {list(harness.adapters)}')
print(f'Gateway bindings:\n{harness.gateway.binding_table()}')
print(f'Prompt layers:\n{harness.prompt.layer_summary()}')
print()

print('='*60)
print('End-to-end: Telegram message through full stack')
print('='*60)
raw_telegram = {
    'message': {
        'message_id': 101,
        'from': {'id': 55555, 'first_name': 'Alice'},
        'text': 'What time is it and what is 2 + 2?',
        'date': int(time.time()),
    }
}
reply = harness.process(raw_telegram, channel_name='telegram')
print(f'\nGateway reply:\n{reply}')

print('\n' + '='*60)
print('End-to-end: CLI message')
print('='*60)
reply2 = harness.process('List 3 benefits of agent gateways.', channel_name='cli')
print(f'\nGateway reply:\n{reply2}')

print(f'\nActive lanes:  {harness.lanes.active_lanes}')
print(f'Delivery WAL:  {harness.delivery._wal_path}')
harness.shutdown()

Initialising GatewayHarness...
[Delivery] Replayed 3 pending message(s) from WAL.
Adapters: ['telegram', 'feishu', 'cli']
Gateway bindings:
Binding table:
  tier=4 (*,*) -> _gateway_agent
Prompt layers:
  BASE           :   185 chars
  IDENTITY       :   214 chars
  SOUL           :   318 chars
  MEMORY         :    73 chars
  SKILLS x0      : (empty)
  TOOLS          :   181 chars
  CONTEXT        : (empty)
  HEARTBEAT      :   361 chars

End-to-end: Telegram message through full stack
  [LaneRouter] New lane: telegram__55555
  [Gateway] tier=4 (*,*) matched (telegram,55555)

Gateway reply:
It's 16:51:10 SGT on 2026-04-21.  
2 + 2 = 4.

End-to-end: CLI message
  [LaneRouter] New lane: cli__local_user
  [Gateway] tier=4 (*,*) matched (cli,local_user)

Gateway reply:
- Centralized security and access control — enforces authentication, authorization, and policies for all agent calls.  
- Smart routing and orchestration — load-balances, routes, and fallbacks across agents for scalability 

# Summary

## Architecture Map

| Chapter | Class / Function | Core Insight |
|---------|-----------------|--------------|
| 1 Intelligence | `PromptAssembler` | 8 named layers; swap files to change personality |
| 2 Channels | `InboundMessage`, adapters | One normalised type across all platforms |
| 3 Gateway | `MessageGateway` | 5-tier binding table; per-peer session isolation |
| 4 Heartbeat | `HeartbeatScheduler`, `CronScheduler` | Agent decides IDLE vs action proactively |
| 5 Delivery | `DeliveryQueue` | WAL-first: disk before API, replay on crash |
| 6 Resilience | `ResilientRunner`, `AuthPool` | 3-layer onion: tool, overflow, auth |
| 7 Concurrency | `LaneRouter`, `Lane` | Named FIFO lanes; per-peer serialisation |
| 8 Capstone | `GatewayHarness` | All layers composed; single `process()` entry point |


```
+-------------------+     +---------------------+
|  Core             |     |  Gateway.            |
|                   |     |                      |
|  run_agent_loop   | <-- |  ResilientRunner     |
|  TodoManager      |     |  GatewayHarness      |
|  FileTaskStore    |     |  MessageGateway      |
|  AutonomousWorker |     |  LaneRouter          |
|  WorktreeManager  |     |  DeliveryQueue       |
|                   |     |  HeartbeatScheduler  |
+-------------------+     +---------------------+
       HOW IT THINKS           HOW IT CONNECTS
```